# 03 – Comparing the groups

**Input:** `results/peptide_features.csv` (from notebook 02)  
**Outputs:** figures in `figures/` and tables `results/comparison_*.csv`

We answer the three questions from the study plan:

* **Q1** – How do *Natural* and *Synthetic* peptides differ in length, charge and amino-acid makeup?
* **Q2** – How do peptides that target different organisms (Gram+, Gram-, fungi, cancer, virus) differ?
* **Q3** – Do peptides active against *mammalian cells* (i.e. cytotoxic) look different from the others?

**Method in one paragraph.** The features are not normally distributed, so we use rank-based tests: **Mann-Whitney U** for two groups and **Kruskal-Wallis** for more than two. With thousands of peptides almost every difference is "significant", so the p-value alone says little. We therefore always report an **effect size**, Cliff's delta (δ): the probability that a random peptide from group A has a higher value than one from group B, minus the reverse. δ = 0 means no difference; δ = ±1 means the groups do not overlap at all.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from scipy.stats import mannwhitneyu, kruskal

df = pd.read_csv("../results/peptide_features.csv")
FIG, RES = "../figures", "../results"

FEATURES = ["length", "net_charge", "charge_density", "hydrophobic_fraction"]
AA = list("ACDEFGHIKLMNPQRSTVWY")
AA_COLS = [f"pct_{a}" for a in AA]
COLORS = {"Natural": "#2a9d8f", "Synthetic": "#e76f51"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print(df.shape)

(18196, 47)


## Helper functions
Small and reusable: one test function, one effect-size label.

In [2]:
def cliffs_delta(a, b):
    """P(a > b) - P(a < b), computed from the Mann-Whitney U statistic."""
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    return 2 * u / (len(a) * len(b)) - 1, p

def size_label(d):
    d = abs(d)
    return "negligible" if d < 0.147 else "small" if d < 0.33 else "medium" if d < 0.474 else "large"

def compare_two(data, group_col, g1, g2, features):
    """Mann-Whitney + Cliff's delta for each feature, g1 vs g2."""
    a, b = data[data[group_col] == g1], data[data[group_col] == g2]
    rows = []
    for f in features:
        d, p = cliffs_delta(a[f], b[f])
        rows.append({"feature": f, f"median_{g1}": a[f].median(), f"median_{g2}": b[f].median(),
                     "cliffs_delta": round(d, 3), "effect": size_label(d), "p_value": p})
    return pd.DataFrame(rows)

# tiny self-test: fully separated groups -> delta = +1 / -1, identical groups -> 0
assert cliffs_delta(pd.Series([5, 6, 7]), pd.Series([1, 2, 3]))[0] == 1
assert cliffs_delta(pd.Series([1, 2, 3]), pd.Series([5, 6, 7]))[0] == -1
assert abs(cliffs_delta(pd.Series([1, 2, 3]), pd.Series([1, 2, 3]))[0]) < 1e-9
print("helpers OK")

helpers OK


# Q1 – Natural vs Synthetic
## 1a. Sample sizes

In [3]:
df["origin"].value_counts()

origin
Synthetic    14957
Natural       3239
Name: count, dtype: int64

## 1b. Length, charge and hydrophobicity

In [4]:
q1 = compare_two(df, "origin", "Natural", "Synthetic", FEATURES)
q1.to_csv(f"{RES}/comparison_q1_origin.csv", index=False)
q1.round({"median_Natural": 2, "median_Synthetic": 2}).style.format({"p_value": "{:.1e}"})

,feature,median_Natural,median_Synthetic,cliffs_delta,effect,p_value
0,length,25.000000,15.000000,0.441000,medium,0.0e+00
1,net_charge,3.000000,4.000000,-0.280000,small,1.6e-139
2,charge_density,0.100000,0.290000,-0.598000,large,0.0e+00
3,hydrophobic_fraction,0.420000,0.440000,-0.079000,negligible,1.4e-12


In [5]:
titles = {"length": "Length (residues)", "net_charge": "Net charge",
          "charge_density": "Charge per residue", "hydrophobic_fraction": "Hydrophobic fraction"}
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, f in zip(axes, FEATURES):
    data = [df.loc[df.origin == o, f] for o in ["Natural", "Synthetic"]]
    bp = ax.boxplot(data, tick_labels=["Natural", "Synthetic"], showfliers=False, patch_artist=True,
                    medianprops={"color": "black"})
    for patch, o in zip(bp["boxes"], ["Natural", "Synthetic"]):
        patch.set_facecolor(COLORS[o]); patch.set_alpha(0.8)
    d = q1.set_index("feature").loc[f]
    ax.set_title(f"{titles[f]}\nδ = {d.cliffs_delta:+.2f} ({d.effect})", fontsize=10)
fig.suptitle("Natural vs Synthetic peptides (outliers hidden)", y=1.02)
plt.tight_layout()
plt.savefig(f"{FIG}/q1_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 1c. Robustness: is it just because natural peptides are longer?
Natural peptides are longer, and length affects the other features. To check, repeat the comparison using only **Short (10–24 aa)** peptides, and separately only peptides **without D-amino acids** (which are much more common in synthetic peptides).

In [6]:
subsets = {
    "all peptides": df,
    "Short (10-24 aa) only": df[df.length_class == "Short (10–24 aa)"],
    "no D-amino acids": df[~df.has_D_aa],
}
rows = []
for name, sub in subsets.items():
    res = compare_two(sub, "origin", "Natural", "Synthetic", ["length", "charge_density", "hydrophobic_fraction"])
    res.insert(0, "subset", name)
    res.insert(1, "n_nat/n_syn", f"{(sub.origin=='Natural').sum()}/{(sub.origin=='Synthetic').sum()}")
    rows.append(res)
robust = pd.concat(rows, ignore_index=True)
robust.to_csv(f"{RES}/comparison_q1_robustness.csv", index=False)
robust.pivot(index="feature", columns="subset", values="cliffs_delta")[list(subsets)]

subset,all peptides,Short (10-24 aa) only,no D-amino acids
feature,,,
charge_density,-0.598,-0.660,-0.576
hydrophobic_fraction,-0.079,0.188,-0.078
length,0.441,0.277,0.465


## 1d. Amino-acid composition
Mean percentage of each amino acid in the two groups, and a Mann-Whitney test per amino acid. Because we run 20 tests, the p-values are multiplied by 20 (Bonferroni correction).

In [7]:
comp = compare_two(df, "origin", "Natural", "Synthetic", AA_COLS)
comp["feature"] = comp["feature"].str.replace("pct_", "")
comp["p_bonferroni"] = (comp["p_value"] * len(AA)).clip(upper=1)
comp = comp.sort_values("cliffs_delta")
comp.to_csv(f"{RES}/comparison_q1_composition.csv", index=False)

order = comp["feature"].tolist()          # sorted from "more in Synthetic" to "more in Natural"
means = df.groupby("origin")[AA_COLS].mean().T
means.index = [i.replace("pct_", "") for i in means.index]
means = means.loc[order]

fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(order)); w = 0.4
ax.bar(x - w/2, means["Natural"], w, label="Natural", color=COLORS["Natural"])
ax.bar(x + w/2, means["Synthetic"], w, label="Synthetic", color=COLORS["Synthetic"])
ax.set_xticks(x); ax.set_xticklabels(order)
ax.set_ylabel("mean % of residues")
ax.set_title("Amino-acid composition (sorted: enriched in Synthetic  →  enriched in Natural)", fontsize=10)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(f"{FIG}/q1_composition.png", dpi=150)
plt.show()

comp[["feature", "median_Natural", "median_Synthetic", "cliffs_delta", "effect", "p_bonferroni"]] \
    .rename(columns={"feature": "amino acid"}).round(3).head(20)

,amino acid,median_Natural,median_Synthetic,cliffs_delta,effect,p_bonferroni
8,K,9.091,15.385,-0.287,small,0.0
14,R,0.000,7.692,-0.260,small,0.0
18,W,0.000,0.000,-0.226,small,0.0
9,L,8.696,9.091,-0.000,negligible,1.0
19,Y,0.000,0.000,0.068,negligible,0.0
13,Q,0.000,0.000,0.091,negligible,0.0
4,F,4.000,0.000,0.097,negligible,0.0
6,H,0.000,0.000,0.100,negligible,0.0
7,I,5.882,5.000,0.116,negligible,0.0
10,M,0.000,0.000,0.131,negligible,0.0


## 1e. Which targets do the two groups hit?
Percentage of peptides in each origin that are active against each target (targets overlap, so columns don't add to 100%). Only peptides with target information are used.

In [8]:
target_cols = ["is_gram_pos", "is_gram_neg", "is_fungus", "is_cancer", "is_virus",
               "is_biofilm", "is_parasite", "is_mammalian_cell"]
t = df[df.has_target_info]
share = (t.groupby("origin")[target_cols].mean() * 100).T
share.index = [i.replace("is_", "") for i in share.index]
share["n_natural"] = t.groupby("origin")[target_cols].sum().T.loc[:, "Natural"].values
share["n_synthetic"] = t.groupby("origin")[target_cols].sum().T.loc[:, "Synthetic"].values
share.to_csv(f"{RES}/comparison_q1_targets.csv")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
im = ax.imshow(share[["Natural", "Synthetic"]].values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Natural", "Synthetic"])
ax.set_yticks(range(len(share))); ax.set_yticklabels(share.index)
for i in range(len(share)):
    for j, o in enumerate(["Natural", "Synthetic"]):
        v = share.loc[share.index[i], o]
        ax.text(j, i, f"{v:.0f}%", ha="center", va="center", color="white" if v > 55 else "black")
ax.set_title("% of peptides active against each target", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG}/q1_target_heatmap.png", dpi=150)
plt.show()
share.round(1)

origin,Natural,Synthetic,n_natural,n_synthetic
gram_pos,87.7,77.3,2818,11360
gram_neg,86.4,81.8,2775,12016
fungus,55.2,24.2,1774,3560
cancer,20.5,16.8,660,2474
virus,4.6,7.5,149,1103
biofilm,2.6,3.6,84,529
parasite,4.7,1.2,152,170
mammalian_cell,54.4,58.3,1746,8559


# Q2 – Peptides with different targets
Target groups overlap (a peptide can hit Gram+, Gram- and fungi at once), so tests that treat them as separate samples would be invalid. To keep the groups **independent**, we only use peptides whose target list is *exactly* one of these:

| Group | Meaning |
|---|---|
| Gram+ only | only Gram-positive bacteria |
| Gram- only | only Gram-negative bacteria |
| Gram+ & Gram- | both bacterial types, nothing else |
| Fungus only | only fungi |
| Virus only | only viruses |
| Cancer only | only cancer cells |

In [9]:
active = [c for c in df.columns if c.startswith("is_")]
t = df[df.has_target_info].copy()
n_targets = t[active].sum(axis=1)

def exclusive_group(r):
    on = frozenset(c for c in active if r[c] == 1)
    return {frozenset(["is_gram_pos"]): "Gram+ only", frozenset(["is_gram_neg"]): "Gram- only",
            frozenset(["is_gram_pos", "is_gram_neg"]): "Gram+ & Gram-",
            frozenset(["is_fungus"]): "Fungus only", frozenset(["is_virus"]): "Virus only",
            frozenset(["is_cancer"]): "Cancer only"}.get(on)

t["target_group"] = t.apply(exclusive_group, axis=1)
t = t.dropna(subset=["target_group"])
GROUPS = ["Gram+ only", "Gram- only", "Gram+ & Gram-", "Fungus only", "Virus only", "Cancer only"]
print(t["target_group"].value_counts().reindex(GROUPS))
print(f"\nOrigin mix inside each group (% Natural):")
print((t.groupby("target_group")["origin"].apply(lambda s: (s == "Natural").mean() * 100)).reindex(GROUPS).round(1))

target_group
Gram+ only        709
Gram- only        692
Gram+ & Gram-    2788
Fungus only       324
Virus only        750
Cancer only       144
Name: count, dtype: int64

Origin mix inside each group (% Natural):
target_group
Gram+ only       13.3
Gram- only        8.8
Gram+ & Gram-    16.8
Fungus only      22.8
Virus only        1.1
Cancer only      48.6
Name: origin, dtype: float64


## 2a. Overall difference between the groups (Kruskal-Wallis)

In [10]:
rows = []
for f in FEATURES:
    h, p = kruskal(*[t.loc[t.target_group == g, f] for g in GROUPS])
    # epsilon-squared: a simple effect size for Kruskal-Wallis (0 = no difference, 1 = complete)
    eps2 = (h - len(GROUPS) + 1) / (len(t) - len(GROUPS))
    rows.append({"feature": f, "H": round(h, 1), "epsilon_sq": round(eps2, 3), "p_value": p,
                 **{f"median_{g}": round(t.loc[t.target_group == g, f].median(), 2) for g in GROUPS}})
q2 = pd.DataFrame(rows)
q2.to_csv(f"{RES}/comparison_q2_targets.csv", index=False)
q2.style.format({"p_value": "{:.1e}"})

,feature,H,epsilon_sq,p_value,median_Gram+ only,median_Gram- only,median_Gram+ & Gram-,median_Fungus only,median_Virus only,median_Cancer only
0,length,147.300000,0.026000,5.0e-30,15.000000,18.000000,18.000000,16.000000,20.000000,8.000000
1,net_charge,977.200000,0.180000,5.2e-209,3.000000,5.000000,4.000000,3.000000,0.000000,0.000000
2,charge_density,955.600000,0.176000,2.4e-204,0.170000,0.280000,0.200000,0.140000,0.000000,0.000000
3,hydrophobic_fraction,268.800000,0.049000,5.1e-56,0.450000,0.310000,0.420000,0.310000,0.400000,0.500000


In [11]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, f in zip(axes, ["length", "charge_density", "hydrophobic_fraction"]):
    data = [t.loc[t.target_group == g, f] for g in GROUPS]
    bp = ax.boxplot(data, tick_labels=GROUPS, showfliers=False, patch_artist=True, medianprops={"color": "black"})
    for patch in bp["boxes"]:
        patch.set_facecolor("#8ecae6")
    ax.set_title(titles[f], fontsize=10)
    ax.tick_params(axis="x", rotation=40)
    for lab in ax.get_xticklabels():
        lab.set_ha("right")
plt.tight_layout()
plt.savefig(f"{FIG}/q2_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 2b. Which pairs of groups differ?
Pairwise Cliff's delta (row group vs column group, positive = row group has higher values). Each cell is a Mann-Whitney comparison; with 15 pairs per feature, only differences that are also at least "small" are worth discussing.

In [12]:
pair_feats = ["length", "charge_density", "hydrophobic_fraction"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw={"wspace": 0.08})
pair_rows = []
for ax, f in zip(axes, pair_feats):
    M = np.zeros((len(GROUPS), len(GROUPS)))
    for i, j in combinations(range(len(GROUPS)), 2):
        d, p = cliffs_delta(t.loc[t.target_group == GROUPS[i], f], t.loc[t.target_group == GROUPS[j], f])
        M[i, j], M[j, i] = d, -d
        pair_rows.append({"feature": f, "group_1": GROUPS[i], "group_2": GROUPS[j],
                          "cliffs_delta": round(d, 3), "effect": size_label(d), "p_bonferroni": min(p * 15, 1)})
    im = ax.imshow(M, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(GROUPS))); ax.set_xticklabels(GROUPS, rotation=40, ha="right", fontsize=8)
    ax.set_yticks(range(len(GROUPS)))
    ax.set_yticklabels(GROUPS if ax is axes[0] else [], fontsize=8)   # labels only on the first panel
    for i in range(len(GROUPS)):
        for j in range(len(GROUPS)):
            if i != j:
                ax.text(j, i, f"{M[i, j]:+.2f}", ha="center", va="center", fontsize=7,
                        color="white" if abs(M[i, j]) > 0.6 else "black")
    ax.set_title(titles[f], fontsize=10)
fig.colorbar(im, ax=axes, shrink=0.8, label="Cliff's delta (row vs column)")
plt.savefig(f"{FIG}/q2_pairwise_delta.png", dpi=150, bbox_inches="tight")
plt.show()
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(f"{RES}/comparison_q2_pairwise.csv", index=False)
print("pairs with at least a medium effect:")
pairs[pairs.cliffs_delta.abs() >= 0.33].sort_values("cliffs_delta", key=abs, ascending=False).head(12)

pairs with at least a medium effect:


,feature,group_1,group_2,cliffs_delta,effect,p_bonferroni
23,charge_density,Gram- only,Cancer only,0.783,large,1.726487e-48
22,charge_density,Gram- only,Virus only,0.738,large,7.372168e-129
26,charge_density,Gram+ & Gram-,Cancer only,0.606,large,1.793965e-33
25,charge_density,Gram+ & Gram-,Virus only,0.577,large,3.488949e-129
19,charge_density,Gram+ only,Cancer only,0.568,large,6.296104e-26
18,charge_density,Gram+ only,Virus only,0.546,large,9.048179e-72
14,length,Virus only,Cancer only,0.546,large,3.781483e-24
28,charge_density,Fungus only,Cancer only,0.534,large,1.912148e-19
27,charge_density,Fungus only,Virus only,0.511,large,2.434364e-39
11,length,Gram+ & Gram-,Cancer only,0.487,large,8.485935e-22


## 2c. Check: is this just Natural vs Synthetic again?
The origin mix differs a lot between the groups (see above: Virus-only peptides are almost all synthetic, Cancer-only about half natural). Since origin has a large effect on charge density (Q1), the Q2 differences could partly be an origin effect. To check, we repeat the medians using **Synthetic peptides only**.

In [13]:
syn = t[t.origin == "Synthetic"]
chk = syn.groupby("target_group")[["length", "charge_density", "hydrophobic_fraction"]].median().reindex(GROUPS)
chk.insert(0, "n", syn.groupby("target_group").size().reindex(GROUPS))
chk.round(2).to_csv(f"{RES}/comparison_q2_synthetic_only.csv")
chk.round(2)

,n,length,charge_density,hydrophobic_fraction
target_group,,,,
Gram+ only,615,15.0,0.21,0.46
Gram- only,631,18.0,0.30,0.31
Gram+ & Gram-,2320,17.0,0.24,0.43
Fungus only,250,12.0,0.20,0.33
Virus only,742,19.5,0.00,0.40
Cancer only,74,12.5,0.07,0.57


# Q3 – Mammalian-cell activity
`is_mammalian_cell = 1` means DBAASP lists mammalian cells among the peptide's targets, i.e. the peptide was found to be active on (cytotoxic/hemolytic to) mammalian cells. **Caveat:** `0` means "not reported", which is not the same as "proven safe" – the peptide may simply not have been tested on mammalian cells. Only peptides with target information are used.

In [14]:
m = df[df.has_target_info].copy()
m["mammalian"] = np.where(m.is_mammalian_cell == 1, "Mammalian-active", "Not reported")
print(m["mammalian"].value_counts(), "\n")

def q3_table(data, label):
    r = compare_two(data, "mammalian", "Mammalian-active", "Not reported", FEATURES)
    r.insert(0, "subset", label)
    return r

q3 = pd.concat([q3_table(m, "all"),
                q3_table(m[m.origin == "Synthetic"], "Synthetic only"),
                q3_table(m[m.origin == "Natural"], "Natural only")], ignore_index=True)
q3.to_csv(f"{RES}/comparison_q3_mammalian.csv", index=False)
q3.round(2).style.format({"p_value": "{:.1e}"})

mammalian
Mammalian-active    10305
Not reported         7600
Name: count, dtype: int64 



,subset,feature,median_Mammalian-active,median_Not reported,cliffs_delta,effect,p_value
0,all,length,16.000000,18.000000,-0.070000,negligible,0.0e+00
1,all,net_charge,4.000000,3.000000,0.180000,small,0.0e+00
2,all,charge_density,0.280000,0.190000,0.230000,small,0.0e+00
3,all,hydrophobic_fraction,0.460000,0.400000,0.200000,small,0.0e+00
4,Synthetic only,length,15.000000,16.000000,-0.050000,negligible,0.0e+00
5,Synthetic only,net_charge,4.000000,4.000000,0.190000,small,0.0e+00
6,Synthetic only,charge_density,0.310000,0.240000,0.230000,small,0.0e+00
7,Synthetic only,hydrophobic_fraction,0.460000,0.420000,0.190000,small,0.0e+00
8,Natural only,length,24.000000,29.000000,-0.180000,small,0.0e+00
9,Natural only,net_charge,3.000000,3.000000,0.070000,negligible,0.0e+00


In [15]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, f in zip(axes, FEATURES):
    data = [m.loc[m.mammalian == g, f] for g in ["Mammalian-active", "Not reported"]]
    bp = ax.boxplot(data, tick_labels=["Mammalian-\nactive", "Not\nreported"], showfliers=False,
                    patch_artist=True, medianprops={"color": "black"})
    for patch, c in zip(bp["boxes"], ["#c1121f", "#adb5bd"]):
        patch.set_facecolor(c); patch.set_alpha(0.8)
    d = q3[(q3.subset == "all") & (q3.feature == f)].iloc[0]
    ax.set_title(f"{titles[f]}\nδ = {d.cliffs_delta:+.2f} ({d.effect})", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG}/q3_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

# Bonus – do the groups separate visually? (PCA)
A quick principal-component analysis on length, charge, charge density, hydrophobic fraction and the 20 composition percentages (all standardised). If Natural and Synthetic overlap heavily, they are not cleanly separable by these simple features.

In [16]:
X = df[FEATURES + AA_COLS].to_numpy(dtype=float)
X = (X - X.mean(axis=0)) / X.std(axis=0)
U, S, Vt = np.linalg.svd(X, full_matrices=False)
pcs = U[:, :2] * S[:2]
var = (S ** 2 / (S ** 2).sum())[:2] * 100

fig, ax = plt.subplots(figsize=(6, 5))
for o in ["Synthetic", "Natural"]:
    mask = (df.origin == o).to_numpy()
    ax.scatter(pcs[mask, 0], pcs[mask, 1], s=4, alpha=0.25, color=COLORS[o], label=o, rasterized=True)
ax.set_xlabel(f"PC1 ({var[0]:.0f}% of variance)"); ax.set_ylabel(f"PC2 ({var[1]:.0f}% of variance)")
ax.legend(markerscale=4, frameon=False)
ax.set_title("PCA of peptide features", fontsize=10)
plt.tight_layout()
plt.savefig(f"{FIG}/pca_origin.png", dpi=150)
plt.show()

top = pd.Series(Vt[0], index=FEATURES + AA_COLS).abs().sort_values(ascending=False).head(5)
print("features driving PC1:", ", ".join(top.index))

features driving PC1: charge_density, net_charge, pct_K, length, pct_E


## One-page summary table

In [17]:
summary = pd.DataFrame([
    {"question": "Q1 Natural vs Synthetic", "feature": r.feature, "effect_size_delta": r.cliffs_delta, "effect": r.effect}
    for r in q1.itertuples()
] + [
    {"question": "Q3 Mammalian-active vs not", "feature": r.feature, "effect_size_delta": r.cliffs_delta, "effect": r.effect}
    for r in q3[q3.subset == "all"].itertuples()
])
summary.to_csv(f"{RES}/comparison_summary.csv", index=False)
summary

,question,feature,effect_size_delta,effect
0,Q1 Natural vs Synthetic,length,0.441,medium
1,Q1 Natural vs Synthetic,net_charge,-0.280,small
2,Q1 Natural vs Synthetic,charge_density,-0.598,large
3,Q1 Natural vs Synthetic,hydrophobic_fraction,-0.079,negligible
4,Q3 Mammalian-active vs not,length,-0.069,negligible
5,Q3 Mammalian-active vs not,net_charge,0.175,small
6,Q3 Mammalian-active vs not,charge_density,0.228,small
7,Q3 Mammalian-active vs not,hydrophobic_fraction,0.199,small


# Findings and limitations

Numbers below are medians unless stated; δ is Cliff's delta (|δ| < 0.15 negligible, < 0.33 small, < 0.47 medium, otherwise large). Everything is "statistically significant" at this sample size, so the discussion relies on effect sizes.

## Q1 – Natural vs Synthetic (3,239 vs 14,957 peptides)
* **Charge per residue is the clearest difference:** 0.10 (natural) vs 0.29 (synthetic), δ = −0.60 (large). It holds when only short peptides are compared (δ = −0.66) and when D-amino-acid peptides are removed (δ = −0.58).
* **Length:** natural peptides are longer, 25 vs 15 residues (δ = +0.44, medium). Raw net charge differs less (3 vs 4, δ = −0.28, small) because natural peptides are longer.
* **Hydrophobic fraction is not a stable difference.** Overall it is nearly identical (0.43 vs 0.44, δ = −0.08), but among short (10–24 aa) peptides the direction flips (natural more hydrophobic, δ = +0.19). The overall comparison is affected by the length difference, so report it with care.
* **Composition:** synthetic peptides are richer in K (mean 17.7% vs 9.7%), R (12.9% vs 5.1%) and W (6.6% vs 1.8%). Natural peptides are richer in G (10.7% vs 5.5%, the only medium effect, δ = +0.42) and, to a smaller extent, S, N, C, P, T and A.
* **Targets:** natural peptides are reported active against fungi far more often (55% vs 24%) and against parasites (4.7% vs 1.2%); Gram+ activity is 88% vs 77%. Virus activity is slightly more common among synthetic peptides (7.5% vs 4.6%).
* **PCA:** the two groups overlap heavily (PC1 and PC2 explain only 14% and 10% of the variance). Synthetic peptides sit further along PC1, which is driven mainly by charge density, but the features do not separate the groups cleanly.

## Q2 – Different targets (exclusive groups only)
* Groups differ mainly in **charge**: Kruskal-Wallis effect size ε² ≈ 0.18 for net charge and charge density, but only 0.03 for length and 0.05 for hydrophobic fraction.
* **Virus-only** and **Cancer-only** peptides have median net charge 0, compared with 3–5 for the antibacterial and antifungal groups (charge-density δ between +0.5 and +0.8 in favour of the antibacterial/antifungal groups).
* **Gram- only** peptides have the highest charge density (0.28) and, together with Fungus-only, the lowest hydrophobic fraction (0.31).
* **Cancer-only** peptides are the shortest (median 8) and the most hydrophobic (0.50), but this group is small (n = 144).
* **Check:** the origin mix differs by group (Virus-only is 1% natural, Cancer-only 49%). Repeating the medians for synthetic peptides only gives the same picture (Virus-only charge density 0.00, Cancer-only 0.07, Gram- only 0.30), so this is not only an origin effect. The synthetic Cancer-only group is small (n = 74).

## Q3 – Mammalian-cell activity
* Peptides reported active on mammalian cells have somewhat higher charge density (0.28 vs 0.19, δ = +0.23), hydrophobic fraction (0.46 vs 0.40, δ = +0.20) and net charge (4 vs 3, δ = +0.18): all small effects. Length is essentially unrelated (δ = −0.07).
* The same small effects appear within synthetic peptides and within natural peptides separately, so they are not caused by the origin mix.
* Cationic and hydrophobic peptides are somewhat more likely to be reported as mammalian-active, but the overlap is large, so these features alone would not predict it well.

## Limitations
* The peptides are not a random sample of all peptides; they are those researchers chose to test and report in DBAASP. Synthetic peptides also outnumber natural ones about 5 to 1.
* About 20% of monomer sequences were removed for containing `X` (unknown residues); the removal rate was similar for both origins (notebook 01).
* Target groups overlap, so Q2 uses only peptides with exactly one target set (or the Gram+ & Gram- pair). This leaves a fraction of the data and very small groups for some targets.
* For Q3, "not reported" is not the same as "not toxic": a peptide may not have been tested on mammalian cells.
* The features are simple sequence-based summaries (no structure, no terminal modifications); the hydrophobic fraction depends on the chosen set of residues.
* These are associations in a curated database, not evidence that a feature *causes* a difference in activity.